In [ ]:
# Install everything you need, including the fix for encrypted PDFs
!pip install transformers==4.45.2 accelerate bitsandbytes PyPDF2 sentence-transformers cryptography pymupdf


In [ ]:
pip install openai

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, json, re
from pathlib import Path
import PyPDF2
import time, re, json
import torch
import fitz  # pymupdf - bulletproof fallback

# ---------- 1. Load Phi‑3.5‑mini (4‑bit on free T4) ----------
model_id = "microsoft/Phi-3.5-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

# ---------- 2. Chunking (1000 chars, 200 overlap – untouched) ----------
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

# ---------- 3. Safe PDF extraction (handles encrypted PDFs) ----------
def extract_text_from_pdf(pdf_path):
    # Try PyPDF2 first
    try:
        with open(pdf_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            if reader.is_encrypted:
                reader.decrypt('')  # attempt empty password
            text = "".join(page.extract_text() or "" for page in reader.pages)
            if text.strip():
                return text
    except Exception as e:
        print(f"PyPDF2 failed for {pdf_path.name} ({e}), trying pymupdf...")

    # Fallback to pymupdf (handles most "encrypted but passwordless" PDFs)
    try:
        doc = fitz.open(pdf_path)
        text = chr(12).join([page.get_text() for page in doc])
        doc.close()
        if text.strip():
            return text
    except Exception as e:
        print(f"Could not read {pdf_path.name}: {e}")
        return ""

# ---------- 4. Load all PDFs and chunk ----------
pdf_dir = "/kaggle/input/datasets/teslaincarnate/scd-pdfs"   # <-- put your folder path here

all_chunks = []
for pdf_file in Path(pdf_dir).glob("*.pdf"):
    text = extract_text_from_pdf(pdf_file)
    if text:
        all_chunks.extend(chunk_text(text, 1000, 200))
    else:
        print(f"Warning: No text extracted from {pdf_file.name}")

print(f"Total chunks: {len(all_chunks)}")



In [ ]:
!pip install -q pymupdf nltk
import fitz  # pymupdf
import re
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

In [ ]:


# ---------- 1. PDF to raw text (no header/footer removal yet, we'll clean content later) ----------
def extract_raw_text(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text() + "\n"
    doc.close()
    return text

# ---------- 2. Aggressive content filter ----------
def clean_body_text(raw_text):
    lines = raw_text.split('\n')
    cleaned_lines = []
    # Patterns to remove entirely
    skip_patterns = [
        r'^\s*Author Manuscript\s*$',                # watermark
        r'^\s*Author Manuscript\s+Author Manuscript\s*$',
        r'^\s*Annu Rev Pathol.*$',                  # journal header
        r'^\s*Published in final edited form.*$',
        r'^\s*doi:\s*\S+',                          # DOI line
        r'^\s*Keywords:.*$',                        # keyword line
        r'^\s*Correspondence.*$',
        r'^\s*\*.*These authors contributed equally.*$',
        r'^\s*prs51@pitt\.edu.*$',                  # specific email
        r'^\s*PMCID:\s*\S+',                        # PubMed Central ID
        r'^\s*NIH-PA Author Manuscript.*$',
        r'^\s*HHS Public Access.*$',
        r'^\s*Author manuscript; available in PMC.*$'
    ]
    for line in lines:
        # Skip empty lines? Keep them for sentence splitting, but remove if only whitespace
        if not line.strip():
            cleaned_lines.append('')   # preserve paragraph breaks
            continue
        # Skip if matches any skip pattern
        if any(re.search(pat, line, re.IGNORECASE) for pat in skip_patterns):
            continue
        # Skip lines that are purely a page number or citation number
        if re.fullmatch(r'\s*\[?\d{1,3}\]?\s*', line):
            continue
        # Skip very short uppercase lines (likely headers like "ABSTRACT")
        if len(line.strip()) < 60 and line.strip().isupper():
            continue
        # Keep the line
        cleaned_lines.append(line.strip())
    return '\n'.join(cleaned_lines)

# ---------- 3. Sentence‑complete chunking (same as before) ----------
def chunk_by_sentences(text, max_chars=1000):
    sentences = sent_tokenize(text)
    chunks = []
    current = ""
    for sent in sentences:
        if len(current) + len(sent) <= max_chars:
            current += " " + sent if current else sent
        else:
            if current:
                chunks.append(current.strip())
            current = sent
    if current:
        chunks.append(current.strip())
    return chunks

# ---------- 4. Process all PDFs ----------
pdf_dir = "/kaggle/input/datasets/teslaincarnate/scd-pdfs"   # <--- CHANGE THIS
all_text = ""
for pdf_file in Path(pdf_dir).glob("*.pdf"):
    raw = extract_raw_text(pdf_file)
    all_text += clean_body_text(raw) + "\n"

chunks = chunk_by_sentences(all_text, max_chars=1000)
print(f"✅ Cleaned {len(chunks)} chunks from {len(all_text)} characters")

In [13]:
import openai, json, re, time, os
from tqdm import tqdm
import random

GROQ_API_KEY = "your-grok-api-key-here"
client = openai.OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

# Use a fast, cheap model – Llama 3.1 8B is good for structured output
MODEL = "llama-3.1-8b-instant"

# Rate limits: free tier = 30 RPM / 6,000 TPM. We'll be safe with 2 requests per second.
REQUESTS_PER_SECOND = 2
DELAY = 1.0 / REQUESTS_PER_SECOND

CHECKPOINT_FILE = "groq_checkpoint.txt"
OUTPUT_FILE = "scd_alpaca.jsonl"

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        start_idx = int(f.read().strip())
    print(f"Resuming from chunk {start_idx}")
else:
    start_idx = 0
    open(OUTPUT_FILE, "w").close()

seen_questions = set()
parse_errors = 0

def get_qa(chunk_text, retries=3):
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You output ONLY a valid JSON array. No markdown, no explanation. Each object has keys 'question' and 'answer'."},
                    {"role": "user", "content": f"Generate 1-3 high-quality Q&A pairs about sickle cell disease from this text:\n\n{chunk_text}\n\nJSON:"}
                ],
                temperature=0.2,
                max_tokens=300
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"Attempt {attempt+1} failed: {e}, waiting {wait:.1f}s")
            time.sleep(wait)
    return None

for i in tqdm(range(start_idx, len(chunks))):
    raw = get_qa(chunks[i])
    if raw is None:
        parse_errors += 1
        continue

    raw = re.sub(r'```(?:json)?\s*|\s*```', '', raw)
    json_match = re.search(r'\[.*\]', raw, re.DOTALL)
    if json_match:
        try:
            qa_pairs = json.loads(json_match.group(0))
            for pair in qa_pairs:
                q = pair.get("question", "").strip()
                a = pair.get("answer", "").strip()
                if q and a and q not in seen_questions:
                    seen_questions.add(q)
                    record = {
                        "instruction": "Answer the following question about sickle cell disease.",
                        "input": q,
                        "output": a
                    }
                    with open(OUTPUT_FILE, "a") as f:
                        f.write(json.dumps(record) + "\n")
        except:
            parse_errors += 1
    else:
        parse_errors += 1

    # Save checkpoint
    with open(CHECKPOINT_FILE, "w") as f:
        f.write(str(i + 1))

    time.sleep(DELAY)  # stay under RPM limit

if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)
print(f"\n✅ Done! {len(seen_questions)} pairs, {parse_errors} parse errors")

Resuming from chunk 1870


100%|██████████| 52/52 [03:11<00:00,  3.68s/it]


✅ Done! 148 pairs, 1 parse errors


In [12]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


In [14]:
import json
with open("scd_alpaca.jsonl") as f:
    lines = f.readlines()
print(f"Total Q&A pairs: {len(lines)}")
if lines:
    sample = json.loads(lines[0])
    print(f"First question: {sample['input'][:100]}")

Total Q&A pairs: 4999
First question: What is the year that sickle cell disease was first discovered?
